# TFT experiment - Walmart Store Sales Forecasting

Person B track (DL + Prophet). Third of the four notebooks in this track,
after `model_experiment_dlinear.ipynb` and `model_experiment_nbeats.ipynb`.

**Library switch (this revision):** previously built on `darts.models.TFTModel`,
which scored Kaggle Public 3615.58 / Private 3779.46 - worse than every
non-DL model in the project. Course feedback on the presentation was that
Darts was the wrong tool for TFT/N-BEATS and a neural-network-native
forecasting library should have been used instead. This notebook is rebuilt
on **Nixtla's `neuralforecast`**, which implements TFT directly rather than
wrapping it. This also removes a real, previously-documented bug class: Darts'
`Scaler` matched fitted per-series transformers to series by *list position*,
which silently corrupted predictions for both N-BEATS and TFT (see
`README.md` SS4.5/4.6). `neuralforecast` keys everything by an explicit
`unique_id` column, so that bug class doesn't exist here, and per-series
target scaling is handled internally by the model (`scaler_type=...`) rather
than by hand.

**Decisions this notebook records for the README:**
- future vs past covariate split - reused as-is from the DLinear/N-BEATS EDA
  finding (checked once against `features.csv` coverage, not re-derived here).
- static covariate scope - `Type` (one-hot) + `Size` (scaled) only, **not**
  raw Store/Dept identity embeddings. `neuralforecast`'s static covariates
  (`stat_exog_list`) are plain numeric columns, not a Darts-style
  `categorical_embedding_sizes` table, so one-hotting ~144 Store/Dept
  categories would be a heavy, untested scope increase for little expected
  gain - the old Darts run's own diagnostics (README SS4.6) already showed
  `Size` dominated static-covariate importance.
- quantile loss (`MQLoss(quantiles=[0.1, 0.5, 0.9])`) - the median is used as
  the point forecast for WMAE/submission, the 10-90% band for interval plots.
- validation strategy - same expanding-window CV (`src/validation.py`) as
  every other model. Attempting the full **3 folds** here (vs. the old
  notebook's compute-cut 1 fold), since `neuralforecast` trains all ~3,300
  series in one global `nf.fit()` call per fold with a fixed step budget,
  rather than Darts' per-series-loop-style fit - expected to be materially
  cheaper. Each fold prints its own wall-clock time; if a real run shows 3
  folds is impractical, drop `N_SPLITS` to 1 and document why, same as the
  old notebook did.
- `h` (forecast horizon) is fixed at model-construction time in
  `neuralforecast` - unlike Darts, a single fitted model can't serve both a
  13-week CV-fold forecast and a 39-week submission forecast. HP-search/CV
  use `h=VAL_WEEKS` (13); the Final stage builds its own model with
  `h=HORIZON` (39).
- **dropped:** `TFTExplainer`-style attention/variable-selection plots -
  Darts-specific, no `neuralforecast` equivalent. Not replaced with a
  substitute.

**MLflow run plan** (under `TFT_Training`, same experiment as the Darts runs
so history stays comparable - each run additionally logs
`library="neuralforecast"` to distinguish them): `TFT_Feature_Selection`,
three `TFT_HP_*` architecture-comparison runs, up to three `TFT_CV_Fold*`
runs, and `TFT_Final`.


##  Init cell (Colab-compatible)

Byte-identical to the DLinear/N-BEATS notebooks' init cell, plus a
`neuralforecast` install alongside `darts[torch]` (DLinear still needs Darts;
only TFT is switching libraries in this revision).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    REPO_URL = "https://github.com/NikaMikeltadze/walmart-sales-forecasting.git"
    REPO_DIR = "/content/walmart-sales-forecasting"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         f"{REPO_DIR}/requirements.txt", "--quiet"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "neuralforecast", "--quiet"], check=True)

    os.chdir(f"{REPO_DIR}/notebooks")

    from google.colab import drive
    drive.mount("/content/drive")

    drive_data_dir = Path("/content/drive/MyDrive/walmart-sales-forecasting/data/raw")
    repo_data_dir = Path(REPO_DIR) / "data" / "raw"
    if drive_data_dir.exists():
        subprocess.run(["cp", "-r", f"{drive_data_dir}/.", str(repo_data_dir)], check=True)
    else:
        raise FileNotFoundError(
            f"Expected Drive data folder not found at {drive_data_dir}. "
            "Create it (or add it as a My Drive shortcut) before running this notebook."
        )

sys.path.append(str(Path.cwd().parent))


##  Imports

In [ ]:
import pickle
import tempfile
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

from sklearn.preprocessing import StandardScaler

import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MQLoss

from pytorch_lightning.loggers import CSVLogger

from src.preprocessing import load_raw_data, WalmartPreprocessor
from src.features import add_temporal_features
from src.evaluation import weighted_mae

pd.set_option("display.max_columns", 50)
pd.set_option("future.no_silent_downcasting", True)


###  Manual credentials override (VS Code / non-Colab-UI runtimes)

`google.colab.userdata` (Colab Secrets) can only be read from the Colab
**browser UI**. When the Colab runtime is driven from VS Code or any other
non-UI frontend it times out (`Secrets can only be fetched when running from
the Colab UI`). This cell sets the DagsHub creds directly instead, and the
credentials cell below skips `userdata` whenever these env vars are already
set.

`getpass` is used so the token is never written into this committed notebook -
run the cell and paste the values when prompted. Leave a prompt blank to fall
through to Colab Secrets / `.env` below (e.g. when you *are* on the Colab UI).


In [ ]:
import os
from getpass import getpass

# Only prompt for values not already set, so re-running cells doesn't re-ask.
# Leave a prompt blank to fall through to Colab Secrets / .env in the next cell.
if not os.environ.get("MLFLOW_TRACKING_USERNAME"):
    _user = getpass("DagsHub username (blank -> use Colab Secrets / .env): ").strip()
    if _user:
        os.environ["MLFLOW_TRACKING_USERNAME"] = _user
if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    _token = getpass("DagsHub token (blank -> use Colab Secrets / .env): ").strip()
    if _token:
        os.environ["MLFLOW_TRACKING_PASSWORD"] = _token


##  Load DagsHub credentials

`MLFLOW_TRACKING_USERNAME`/`MLFLOW_TRACKING_PASSWORD` are never hardcoded in
this notebook (it gets committed to the shared repo, so a hardcoded token
would leak).

- On the Colab UI: read from Colab secrets - add `DAGSHUB_USERNAME` and
  `DAGSHUB_TOKEN` via the key icon in the left sidebar, and enable
  "Notebook access" for both. Same approach as every other notebook.
- From VS Code / non-UI runtimes: use the manual-override cell above.
- Locally: falls back to a gitignored `.env` in the repo root.


In [ ]:
if os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    # Already provided (e.g. by the manual-override cell above when driving the
    # Colab runtime from VS Code, where google.colab.userdata would time out).
    # Note: userdata.get(...) must NOT be evaluated in this case - it blocks for
    # ~minutes and raises when there is no Colab browser UI to answer it.
    creds_source = "pre-set environment variables"
elif IS_COLAB:
    from google.colab import userdata

    os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
    creds_source = "Colab secrets (DAGSHUB_USERNAME / DAGSHUB_TOKEN)"
else:
    from dotenv import load_dotenv

    env_path = Path.cwd().parent / ".env"
    load_dotenv(env_path)
    creds_source = str(env_path)

assert os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"), (
    f"MLFLOW_TRACKING_USERNAME/PASSWORD not set (tried: {creds_source}). "
    "On the Colab UI: add DAGSHUB_USERNAME and DAGSHUB_TOKEN as Colab secrets "
    "(key icon in the left sidebar) and enable notebook access for both. "
    "From VS Code / non-UI runtimes: run the manual-override cell above. "
    "Locally: create a .env in the repo root with MLFLOW_TRACKING_USERNAME=... "
    "and MLFLOW_TRACKING_PASSWORD=..."
)
print("MLflow credentials loaded from:", creds_source)


##  MLflow tracking store

Shared DagsHub MLflow server - the single source of truth for cross-model
WMAE comparison and the Model Registry, so all models log here rather than to
a per-notebook local store. Same experiment name as the previous Darts-based
runs (`TFT_Training`) so history stays comparable; each run below logs
`library="neuralforecast"` as a param to distinguish new runs from old ones.
Does not silently fall back to a local store if auth fails - that would
desync TFT's runs from the rest of the team's.


In [ ]:
MLFLOW_TRACKING_URI = "https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

try:
    mlflow.set_experiment("TFT_Training")
    mlflow.MlflowClient().search_experiments(max_results=1)  # force a network round trip now
except Exception as e:
    raise RuntimeError(
        "Could not authenticate to the DagsHub MLflow server at "
        f"{MLFLOW_TRACKING_URI}. Set MLFLOW_TRACKING_USERNAME and "
        "MLFLOW_TRACKING_PASSWORD (a DagsHub access token) as environment "
        "variables, then re-run this cell. Not falling back to local "
        "./mlruns or sqlite - that would desync from the rest of the team's runs."
    ) from e

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("TFT_Training").name)


##  Load, merge, clean

Reuses `load_raw_data` / `WalmartPreprocessor` rather than reimplementing
loading logic. `WalmartPreprocessor.fit()` loads and caches `features_` /
`stores_` internally - `stores_` (Store, Type, Size) is what the
static-covariate section below builds on.


In [ ]:
raw_data = load_raw_data(data_dir="../data/raw")

train_raw = raw_data["train"]
test_raw = raw_data["test"]

preprocessor = WalmartPreprocessor(data_dir="../data/raw")
preprocessor.fit(train_raw)

train_clean = preprocessor.transform(train_raw)
test_clean = preprocessor.transform(test_raw)

train_feat = add_temporal_features(train_clean)
test_feat = add_temporal_features(test_clean)

print(train_feat.shape, test_feat.shape)
train_feat.head()


##  Covariate coverage decision

Same check already established in the DLinear/N-BEATS/old-TFT notebooks -
reused here rather than re-derived, since it's a property of `features.csv`,
not of the model. A column only becomes a *future* covariate if it's
non-null for every week of the test horizon (2012-11-02 to 2013-07-26).


In [ ]:
test_dates = pd.date_range("2012-11-02", "2013-07-26", freq="W-FRI")

macro_cols = ["Temperature", "Fuel_Price", "CPI", "Unemployment",
              "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]

features_lookup = preprocessor.features_

coverage = {}
for col in macro_cols:
    non_null_dates = set(features_lookup.loc[features_lookup[col].notna(), "Date"])
    coverage[col] = set(test_dates).issubset(non_null_dates)

coverage_df = pd.Series(coverage, name="covers_full_test_range")
print(coverage_df)

calendar_future_cols = ["IsHoliday", "week_of_year", "month", "year",
                         "days_to_super_bowl", "days_to_labor_day",
                         "days_to_thanksgiving", "days_to_christmas"]

FUTURE_COVARIATE_COLS = calendar_future_cols + [c for c in coverage_df.index if coverage_df[c]]
PAST_COVARIATE_COLS = [c for c in coverage_df.index if not coverage_df[c]]
all_covariate_cols = FUTURE_COVARIATE_COLS + PAST_COVARIATE_COLS

print("Future covariates:", FUTURE_COVARIATE_COLS)
print("Past covariates:", PAST_COVARIATE_COLS)


##  Static covariates (`Type` + `Size` only - see intro markdown for why)

`neuralforecast`'s `stat_exog_list` takes plain numeric columns (no
Darts-style `categorical_embedding_sizes` table), so `Type` is one-hot
encoded and `Size` is scaled with `StandardScaler` (fit on the store table,
which has no train/test split of its own). One static row is built per
`unique_id` (`"{Store}_{Dept}"`), covering every (Store, Dept) pair seen in
train **or** test, so the pipeline's `.predict()` never hits a missing
static-covariate lookup.


In [ ]:
stores_df = preprocessor.stores_.copy()

type_dummies = pd.get_dummies(stores_df["Type"].astype(str), prefix="Type").astype(float)
TYPE_COLS = sorted(type_dummies.columns.tolist())

size_scaler = StandardScaler()
stores_df["Size_Scaled"] = size_scaler.fit_transform(stores_df[["Size"]]).astype(float)

stores_static = pd.concat([stores_df[["Store", "Size_Scaled"]], type_dummies], axis=1)
STATIC_COLS = TYPE_COLS + ["Size_Scaled"]

all_pairs = (
    pd.concat([train_feat[["Store", "Dept"]], test_feat[["Store", "Dept"]]])
    .drop_duplicates()
    .reset_index(drop=True)
)

static_df = all_pairs.merge(stores_static, on="Store", how="left")
static_df["unique_id"] = static_df["Store"].astype(str) + "_" + static_df["Dept"].astype(str)
static_df = static_df[["unique_id"] + STATIC_COLS]

print("Static covariates:", STATIC_COLS)
print(f"static_df: {static_df.shape[0]} (Store, Dept) series")
static_df.head()


##  Model constants and compute budget

**Runtime budget (kept deliberately tight, same spirit as the old
Darts-based notebook):** `neuralforecast` trains ALL series inside one
global `nf.fit()` call per model, so the step budgets below
(`HP_MAX_STEPS`/`CV_MAX_STEPS`/`FINAL_MAX_STEPS`) are **total gradient steps
across the whole ~3,300-series panel**, not per-series like Darts'
`n_epochs` x `max_samples_per_ts` - epoch time no longer scales with total
history the way it did in Darts. `start_padding_enabled=True` on every model
below zero-pads any (Store, Dept) series shorter than
`input_size + h`, so no series needs to be dropped from training the way the
old notebook's `MIN_REQUIRED_LENGTH` filter did.

`h` (forecast horizon) is fixed per model at construction time - CV/HP models
use `h=VAL_WEEKS` (13, matching every other model's fold convention), the
Final model uses `h=HORIZON` (39, the full test horizon).


In [ ]:
FREQ = "W-FRI"
inferred_freq = pd.infer_freq(sorted(train_feat["Date"].unique()))
print("Inferred frequency from data:", inferred_freq)
assert inferred_freq in ("W-FRI", None), f"Unexpected frequency: {inferred_freq}"

INPUT_SIZE = 52   # one year of history
HORIZON = 39      # covers the full test horizon - test is ~39 weeks past train
VAL_WEEKS = 13    # CV fold validation window - same convention as every other model

QUANTILES = [0.1, 0.5, 0.9]

GPU_AVAILABLE = torch.cuda.is_available()
print("GPU available:", GPU_AVAILABLE)

# Step budgets - see markdown above for why these are NOT per-series like Darts' n_epochs.
HP_MAX_STEPS = 80        # ceiling for the quick 3-way architecture comparison
CV_MAX_STEPS = 150       # ceiling per CV fold
FINAL_MAX_STEPS = 400    # ceiling for the single final fit on full history

BATCH_SIZE_SEARCH = 256  # HP search + CV folds
BATCH_SIZE_FINAL = 512


def make_trainer_kwargs(logger=None):
    """Keyword args forwarded through TFT's **trainer_kwargs to the underlying
    pl.Trainer, so GPU selection is consistent everywhere below."""
    kwargs = {"accelerator": "gpu" if GPU_AVAILABLE else "cpu", "enable_progress_bar": False}
    if logger is not None:
        kwargs["logger"] = logger
    if GPU_AVAILABLE:
        kwargs["devices"] = 1
    return kwargs


def median_column(preds: pd.DataFrame) -> str:
    """Name of the median-quantile column in a neuralforecast predict() output -
    looked up by suffix rather than hardcoded, since it depends on the model's alias."""
    candidates = [c for c in preds.columns if c.endswith("-median")]
    assert len(candidates) == 1, f"Expected exactly one '-median' column, got {candidates}"
    return candidates[0]


print("Columns fed as hist_exog_list (past covariates):", PAST_COVARIATE_COLS)
print("Columns fed as futr_exog_list (future covariates):", FUTURE_COVARIATE_COLS)
print("Columns fed as stat_exog_list (static covariates):", STATIC_COLS)


##  Build the long-format panel (gap-interpolated target, scaled covariates)

`neuralforecast` takes one long DataFrame (`unique_id`, `ds`, `y`, plus
exogenous columns) rather than Darts' per-series `TimeSeries` list. Target
gaps are linearly interpolated (interior missing weeks, not 0-sales weeks) -
same as the Darts notebooks. Covariates are built once per Store (they don't
vary by Dept) over the full train+test date range, ffill/bfill'd, then
merged onto every (Store, Dept) row - immune to any single department's
short test presence, same intent as the old `build_store_level_covariates`.

Numeric covariates (not the target) are scaled with a single `StandardScaler`
fit on **train dates only**, matching the old notebook's `df_scaler`. The
target `y` is **not** manually scaled - `TFT(scaler_type="standard")` below
normalizes/denormalizes each series internally, keyed by `unique_id` identity,
which is exactly the design flaw that caused the Darts `Scaler` positional
bug in the first place (see intro markdown).


In [ ]:
def build_covariate_lookup(full_df, covariate_cols, global_start, global_end, freq=FREQ):
    """One row per (Store, Date) covariate snapshot, ffill/bfill'd per store,
    spanning the full global range so it always covers any forecast horizon."""
    store_level = full_df.drop_duplicates(subset=["Store", "Date"])[["Store", "Date"] + covariate_cols]
    full_range = pd.date_range(global_start, global_end, freq=freq)
    filled = []
    for store, g in store_level.groupby("Store"):
        g = g.sort_values("Date").set_index("Date").reindex(full_range)
        g.index.name = "Date"
        g[covariate_cols] = g[covariate_cols].ffill().bfill()
        g["Store"] = store
        filled.append(g.reset_index())
    lookup = pd.concat(filled, ignore_index=True)
    return lookup.rename(columns={"Date": "ds"})


def build_target_long(df, freq=FREQ):
    """Long (unique_id, ds, y, Store, Dept) frame, one row per (Store, Dept, Date).
    Target gaps are linearly interpolated (genuine missing weeks, not 0-sales weeks)."""
    rows = []
    for (store, dept), g in df.groupby(["Store", "Dept"]):
        g = g.sort_values("Date")
        full_range = pd.date_range(g["Date"].min(), g["Date"].max(), freq=freq)
        y = g.set_index("Date")["Weekly_Sales"].reindex(full_range)
        y = y.interpolate(method="linear", limit_direction="both")
        rows.append(pd.DataFrame({
            "unique_id": f"{store}_{dept}",
            "ds": full_range,
            "y": y.values,
            "Store": store,
            "Dept": dept,
        }))
    return pd.concat(rows, ignore_index=True)


numeric_cols = all_covariate_cols
full_feat_df = pd.concat([train_feat, test_feat], ignore_index=True).drop_duplicates(
    subset=["Store", "Dept", "Date"]
)
df_scaler = StandardScaler()
train_dates = train_feat["Date"].unique()
df_scaler.fit(full_feat_df.loc[full_feat_df["Date"].isin(train_dates), numeric_cols])
full_feat_scaled_df = full_feat_df.copy()
full_feat_scaled_df[numeric_cols] = df_scaler.transform(full_feat_df[numeric_cols])

GLOBAL_START = full_feat_scaled_df["Date"].min()
GLOBAL_END = full_feat_scaled_df["Date"].max()
print(f"Covariates span {GLOBAL_START} -> {GLOBAL_END}")

covariate_lookup = build_covariate_lookup(full_feat_scaled_df, all_covariate_cols, GLOBAL_START, GLOBAL_END)
target_long = build_target_long(train_feat)

panel_df = target_long.merge(covariate_lookup, on=["Store", "ds"], how="left")
n_nan_cov = panel_df[all_covariate_cols].isna().sum().sum()
assert n_nan_cov == 0, f"panel_df: {n_nan_cov} NaN covariate values after merge - stop"

print(f"Built panel_df: {panel_df.shape[0]} rows, {panel_df['unique_id'].nunique()} (Store, Dept) series")
panel_df.head()


## Time-based CV: expanding-window splits

Uses the real shared `src/validation.py` splitter (`expanding_window_splits`)
- same one every other model in the project uses. Attempting the full
**3 folds** (vs. the old Darts notebook's compute-cut 1 fold) - see intro
markdown for why this is expected to be cheaper here. Each fold below prints
its own wall-clock fit+predict time; if a real run shows 3 folds is
impractical, drop `N_SPLITS` to 1 and document why, same spirit as the old
notebook's compute-driven scope reduction.


In [ ]:
from src.validation import expanding_window_splits, describe_split

N_SPLITS = 3
MIN_TRAIN_WEEKS = 52

splits = expanding_window_splits(
    train_feat["Date"], n_splits=N_SPLITS, val_weeks=VAL_WEEKS, min_train_weeks=MIN_TRAIN_WEEKS
)
assert len(splits) == N_SPLITS, "history too short for the requested number of folds"

for i, split in enumerate(splits):
    print(f"fold {i}:", describe_split(split))


##  Log covariate/static-covariate decisions (`TFT_Feature_Selection`)

Lightweight run - just records the covariate split, static-covariate scope,
and library choice as params. No model fit happens here (actual training
happens in the HP-search, CV, and Final stages below).


In [ ]:
with mlflow.start_run(run_name="TFT_Feature_Selection"):
    mlflow.log_params({
        "library": "neuralforecast",
        "future_covariates": ",".join(FUTURE_COVARIATE_COLS) if FUTURE_COVARIATE_COLS else "none",
        "past_covariates": ",".join(PAST_COVARIATE_COLS),
        "static_covariates": ",".join(STATIC_COLS),
        "static_covariate_scope": "Type (one-hot) + Size (scaled) - no Store/Dept identity embeddings",
        "likelihood": f"MQLoss({QUANTILES})",
        "scaling": "StandardScaler (covariates only, fit on train) + TFT(scaler_type='standard') for the target internally, per-series by unique_id",
        "target_gap_fill": "linear_interpolation (interior gaps only - genuine 0-sales weeks are untouched)",
        "covariate_gap_fill": "ffill/bfill, built once per Store (covariates are Dept-invariant)",
        "n_series_total": panel_df["unique_id"].nunique(),
        "input_size": INPUT_SIZE,
        "horizon_cv": VAL_WEEKS,
        "horizon_final": HORIZON,
        "start_padding_enabled": True,
    })

print("Logged TFT_Feature_Selection.")


##  Shared helper: build one CV fold's frames

Used by both the hyperparameter comparison below and the full CV loop, so
the exact same fold-construction logic backs both.


In [ ]:
def split_long(panel, split):
    """Apply a Split (src/validation.py) to the long panel_df. Returns (train_long, val_long)."""
    train_mask = panel["ds"] <= split.train_end
    if split.train_start is not None:
        train_mask &= panel["ds"] >= split.train_start
    val_mask = (panel["ds"] >= split.val_start) & (panel["ds"] <= split.val_end)
    return panel.loc[train_mask].copy(), panel.loc[val_mask].copy()


def fit_predict_fold(split, max_steps, batch_size, hidden_size=128, n_head=4, dropout=0.1, logger=None):
    """Fit a fresh TFT on one fold's train frame, forecast its val window, and
    return (wmae, mae, scored_df, elapsed_seconds). Never reuses a model object
    whose .fit() already ran - a fresh TFT/NeuralForecast is built every call."""
    fold_train, fold_val = split_long(panel_df, split)

    futr_df = fold_val[["unique_id", "ds"] + FUTURE_COVARIATE_COLS].copy()

    t0 = time.time()
    model = TFT(
        h=VAL_WEEKS,
        input_size=INPUT_SIZE,
        stat_exog_list=STATIC_COLS,
        hist_exog_list=PAST_COVARIATE_COLS,
        futr_exog_list=FUTURE_COVARIATE_COLS,
        loss=MQLoss(quantiles=QUANTILES),
        max_steps=max_steps,
        batch_size=batch_size,
        hidden_size=hidden_size,
        n_head=n_head,
        dropout=dropout,
        scaler_type="standard",
        random_seed=42,
        start_padding_enabled=True,
        **make_trainer_kwargs(logger=logger),
    )
    nf = NeuralForecast(models=[model], freq=FREQ)
    nf.fit(
        df=fold_train[["unique_id", "ds", "y"] + PAST_COVARIATE_COLS + FUTURE_COVARIATE_COLS],
        static_df=static_df,
    )
    preds = nf.predict(futr_df=futr_df)
    elapsed = time.time() - t0

    med_col = median_column(preds)
    scored = preds.merge(
        fold_val[["unique_id", "ds", "y", "Store", "Dept", "IsHoliday"]],
        on=["unique_id", "ds"], how="inner",
    )
    scored = scored.rename(columns={med_col: "Weekly_Sales_Pred", "y": "Weekly_Sales"})

    wmae = weighted_mae(scored["Weekly_Sales"].values, scored["Weekly_Sales_Pred"].values, scored["IsHoliday"].values)
    mae = float(np.mean(np.abs(scored["Weekly_Sales"].values - scored["Weekly_Sales_Pred"].values)))
    return wmae, mae, scored, elapsed, nf


##  Hyperparameter comparison (`TFT_HP_Baseline` / `TFT_HP_Wide` / `TFT_HP_Deep`)

A quick, tight-step-budget comparison of three architecture sizes on the most
recent CV fold (`splits[-1]`, the fold closest to the final fit's actual
history), to pick a configuration before committing to the full CV below.
Each candidate gets its own MLflow run so the comparison is inspectable
run-by-run in the DagsHub UI, not just as a printed table.


In [ ]:
HP_CONFIGS = [
    {"name": "Baseline", "hidden_size": 64, "n_head": 4, "dropout": 0.1},
    {"name": "Wide", "hidden_size": 128, "n_head": 4, "dropout": 0.1},
    {"name": "Deep", "hidden_size": 64, "n_head": 8, "dropout": 0.2},
]

hp_split = splits[-1]
hp_results = []

for cfg in HP_CONFIGS:
    cfg_kwargs = {k: v for k, v in cfg.items() if k != "name"}
    with mlflow.start_run(run_name=f"TFT_HP_{cfg['name']}"):
        mlflow.log_params({
            "library": "neuralforecast",
            "hp_candidate": cfg["name"],
            "max_steps": HP_MAX_STEPS,
            "batch_size": BATCH_SIZE_SEARCH,
            "fold": describe_split(hp_split),
            "input_size": INPUT_SIZE,
            "h": VAL_WEEKS,
            **cfg_kwargs,
        })

        wmae_hp, mae_hp, _, elapsed_hp, _ = fit_predict_fold(
            hp_split, max_steps=HP_MAX_STEPS, batch_size=BATCH_SIZE_SEARCH, **cfg_kwargs
        )
        mlflow.log_metric("wmae", wmae_hp)
        mlflow.log_metric("mae", mae_hp)
        mlflow.log_metric("fit_predict_seconds", elapsed_hp)

    hp_results.append({"name": cfg["name"], "wmae": wmae_hp, "mae": mae_hp, "seconds": elapsed_hp, **cfg_kwargs})
    print(f"{cfg['name']}: WMAE={wmae_hp:.2f}  MAE={mae_hp:.2f}  ({elapsed_hp:.1f}s)")

hp_results_df = pd.DataFrame(hp_results).sort_values("wmae")
print(hp_results_df)

BEST_HP = hp_results_df.iloc[0].to_dict()
BEST_HP_KWARGS = {
    "hidden_size": int(BEST_HP["hidden_size"]),
    "n_head": int(BEST_HP["n_head"]),
    "dropout": float(BEST_HP["dropout"]),
}
print("Selected architecture for CV/Final:", BEST_HP_KWARGS)


##  Multi-fold expanding-window CV (`TFT_CV_Fold0` / `TFT_CV_Fold1` / `TFT_CV_Fold2`)

Refits a fresh TFT per fold with `BEST_HP_KWARGS` from the search above,
forecasts the fold's 13-week validation window, and scores with the
competition's `weighted_mae`. Each fold is logged as its own MLflow run
(rather than one run with three metrics) so all folds are individually
inspectable, and a per-fold training-loss curve (via `CSVLogger`) is logged
as an artifact on each. **Wall-clock time is printed and logged per fold** -
if a real run shows this is impractical within the available compute budget,
drop `N_SPLITS` to 1 above and re-run, documenting why (same spirit as the
old Darts notebook's own compute-driven scope reduction).


In [ ]:
fold_results = []
scored_folds = []

for i, split in enumerate(splits):
    csv_logger = CSVLogger(save_dir="../submissions/_tft_logs", name=f"cv_fold_{i}")

    with mlflow.start_run(run_name=f"TFT_CV_Fold{i}"):
        mlflow.log_params({
            "library": "neuralforecast",
            "cv_strategy": "expanding_window",
            "fold_index": i,
            "n_splits": N_SPLITS,
            "val_weeks": VAL_WEEKS,
            "min_train_weeks": MIN_TRAIN_WEEKS,
            "max_steps": CV_MAX_STEPS,
            "batch_size": BATCH_SIZE_SEARCH,
            "input_size": INPUT_SIZE,
            "future_covariates": ",".join(FUTURE_COVARIATE_COLS) if FUTURE_COVARIATE_COLS else "none",
            "past_covariates": ",".join(PAST_COVARIATE_COLS),
            **BEST_HP_KWARGS,
            **describe_split(split),
        })

        wmae_fold, mae_fold, scored_fold, elapsed_fold, _ = fit_predict_fold(
            split, max_steps=CV_MAX_STEPS, batch_size=BATCH_SIZE_SEARCH, logger=csv_logger, **BEST_HP_KWARGS
        )
        scored_fold["fold"] = i

        mlflow.log_metric("wmae", wmae_fold)
        mlflow.log_metric("mae", mae_fold)
        mlflow.log_metric("fit_predict_seconds", elapsed_fold)

        try:
            loss_history = pd.read_csv(f"{csv_logger.log_dir}/metrics.csv")
            loss_col = "train_loss_epoch" if "train_loss_epoch" in loss_history else "train_loss"
            loss_plot_path = f"../submissions/_tft_train_loss_fold{i}.png"
            plt.figure(figsize=(6, 4))
            loss_curve = loss_history.dropna(subset=[loss_col]) if loss_col in loss_history else None
            if loss_curve is not None and not loss_curve.empty:
                plt.plot(loss_curve["step"], loss_curve[loss_col], marker="o")
            plt.title(f"TFT CV Fold {i} - training loss")
            plt.xlabel("Step")
            plt.ylabel("Train loss")
            plt.tight_layout()
            plt.savefig(loss_plot_path, dpi=120)
            plt.close()
            mlflow.log_artifact(loss_plot_path)
        except Exception as e:
            print(f"fold {i}: could not read/plot loss history ({e})")

    fold_results.append({**describe_split(split), "wmae": wmae_fold, "mae": mae_fold, "seconds": elapsed_fold})
    scored_folds.append(scored_fold)
    print(f"fold {i}: WMAE={wmae_fold:.2f}  MAE={mae_fold:.2f}  ({elapsed_fold:.1f}s)")

wmae_mean = float(np.mean([r["wmae"] for r in fold_results]))
wmae_std = float(np.std([r["wmae"] for r in fold_results]))
mae_mean = float(np.mean([r["mae"] for r in fold_results]))

scored = pd.concat(scored_folds, ignore_index=True)
print(f"\nCV WMAE: {wmae_mean:.2f} (+/- {wmae_std:.2f})")
print(f"CV MAE:  {mae_mean:.2f}")
print(f"Total CV wall-clock time: {sum(r['seconds'] for r in fold_results):.1f}s across {len(fold_results)} fold(s)")


##  Diagnostics

Aggregate actual vs predicted, error distribution, holiday vs non-holiday
WMAE split, per-series WMAE, and WMAE broken down by store `Type` - a direct
readout of whether the static-covariate encoder is actually picking up
store-level differences. Pools predictions across all CV folds (`scored`
from the cell above).


In [ ]:
scored["error"] = scored["Weekly_Sales_Pred"] - scored["Weekly_Sales"]
scored["abs_error"] = scored["error"].abs()
scored["weight"] = scored["IsHoliday"].map({True: 5, False: 1})
scored = scored.merge(stores_df[["Store", "Type"]], on="Store", how="left")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

agg = scored.groupby("ds")[["Weekly_Sales", "Weekly_Sales_Pred"]].sum().sort_index()
axes[0, 0].plot(agg.index, agg["Weekly_Sales"], label="Actual", marker="o")
axes[0, 0].plot(agg.index, agg["Weekly_Sales_Pred"], label="Predicted", marker="o")
axes[0, 0].set_title("Aggregate Weekly Sales: Actual vs Predicted (Pooled CV Folds)")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Total Weekly Sales ($)")
axes[0, 0].legend()
axes[0, 0].tick_params(axis="x", rotation=45)

axes[0, 1].hist(scored["error"], bins=80, color="steelblue")
axes[0, 1].axvline(0, color="red", linestyle="--")
axes[0, 1].set_title("Prediction Error Distribution")
axes[0, 1].set_xlabel("Predicted - Actual")
axes[0, 1].set_ylabel("Count")

by_holiday = scored.groupby("IsHoliday")[["abs_error", "weight"]].apply(
    lambda g: (g["abs_error"] * g["weight"]).sum() / g["weight"].sum()
)
axes[1, 0].bar(["Non-Holiday", "Holiday"], by_holiday.values, color=["steelblue", "orange"])
axes[1, 0].set_title("WMAE: Holiday vs Non-Holiday Weeks")
axes[1, 0].set_ylabel("Weighted MAE")


def series_wmae(g):
    w = g["IsHoliday"].map({True: 5, False: 1})
    return (g["abs_error"] * w).sum() / w.sum()


per_series = scored.groupby(["Store", "Dept"])[["IsHoliday", "abs_error"]].apply(series_wmae).sort_values(ascending=False)
axes[1, 1].hist(per_series, bins=80, color="seagreen")
axes[1, 1].set_title("Per-Series WMAE Distribution")
axes[1, 1].set_xlabel("WMAE per (Store, Dept)")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("../submissions/_tft_diagnostics.png", dpi=120)
plt.show()

by_type = scored.groupby("Type")[["abs_error", "weight"]].apply(
    lambda g: (g["abs_error"] * g["weight"]).sum() / g["weight"].sum()
).sort_index()

plt.figure(figsize=(6, 4))
plt.bar(by_type.index.astype(str), by_type.values, color=["#4c72b0", "#dd8452", "#55a868"])
plt.title("WMAE by Store Type (static covariate)")
plt.xlabel("Store Type")
plt.ylabel("Weighted MAE")
plt.tight_layout()
plt.savefig("../submissions/_tft_wmae_by_type.png", dpi=120)
plt.show()

print("Top 10 worst-performing series (highest WMAE):")
print(per_series.head(10))
print("\nWMAE by store Type:")
print(by_type)


##  Representative-series forecast plots with quantile bands

Best- and worst-WMAE series from the diagnostics above, each showing
history, actual, the median forecast, and the 10-90% quantile band from the
probabilistic TFT output.


In [ ]:
def plot_series_forecast(store, dept, split, ax, title_prefix):
    uid = f"{store}_{dept}"
    _, _, fold_scored, _, nf_fold = fit_predict_fold(
        split, max_steps=CV_MAX_STEPS, batch_size=BATCH_SIZE_SEARCH, **BEST_HP_KWARGS
    )

    hist_df = panel_df.loc[(panel_df["unique_id"] == uid) & (panel_df["ds"] <= split.train_end)]
    series_scored = fold_scored.loc[fold_scored["unique_id"] == uid].sort_values("ds")

    lo_col = [c for c in series_scored.columns if c.endswith("-lo-80.0")]
    hi_col = [c for c in series_scored.columns if c.endswith("-hi-80.0")]

    ax.plot(hist_df["ds"], hist_df["y"], color="gray", label="History")
    ax.plot(series_scored["ds"], series_scored["Weekly_Sales"], color="black", marker="o", label="Actual")
    ax.plot(series_scored["ds"], series_scored["Weekly_Sales_Pred"], color="steelblue", marker="x", label="Median forecast")
    if lo_col and hi_col:
        ax.fill_between(
            series_scored["ds"], series_scored[lo_col[0]], series_scored[hi_col[0]],
            color="steelblue", alpha=0.2, label="10-90% band",
        )
    ax.axvline(split.train_end, color="red", linestyle="--", linewidth=1)
    ax.set_title(f"{title_prefix}: Store {store}, Dept {dept}")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=45)


best_key = per_series.index[-1]
worst_key = per_series.index[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_series_forecast(best_key[0], best_key[1], splits[-1], axes[0], "Best (lowest WMAE)")
plot_series_forecast(worst_key[0], worst_key[1], splits[-1], axes[1], "Worst (highest WMAE)")
plt.tight_layout()
plt.savefig("../submissions/_tft_representative_forecasts.png", dpi=120)
plt.show()


##  Required `.predict()` wrapper (`NeuralForecastPipeline`)

Every model needs a pipeline-shaped wrapper that takes raw, unpreprocessed
`test.csv`-shaped data and handles everything internally, so
`model_inference.ipynb` can call it directly. Contract is identical to the
old `DartsForecastPipeline`: raw test df in, `DataFrame[Id, Weekly_Sales]`
out. No per-series `Scaler`/`inverse_transform` bookkeeping needed here -
`neuralforecast` denormalizes each series internally by `unique_id` before
returning predictions.


In [ ]:
class NeuralForecastPipeline:
    def __init__(self, nf_model, preprocessor, df_scaler, numeric_cols,
                 future_covariate_cols, static_df, train_history_df,
                 freq="W-FRI"):
        self.nf_model = nf_model
        self.preprocessor = preprocessor
        self.df_scaler = df_scaler
        self.numeric_cols = numeric_cols
        self.future_covariate_cols = future_covariate_cols
        self.static_df = static_df
        self.train_history_df = train_history_df
        self.freq = freq

    def predict(self, raw_test_df):
        test_clean = self.preprocessor.transform(raw_test_df)
        test_feat_local = add_temporal_features(test_clean)

        futr_local = test_feat_local.copy()
        futr_local["unique_id"] = futr_local["Store"].astype(str) + "_" + futr_local["Dept"].astype(str)
        futr_local["ds"] = futr_local["Date"]
        futr_local[self.numeric_cols] = self.df_scaler.transform(futr_local[self.numeric_cols])
        futr_df = futr_local[["unique_id", "ds"] + self.future_covariate_cols]

        preds = self.nf_model.predict(futr_df=futr_df)
        med_col = median_column(preds)
        preds = preds.rename(columns={med_col: "Weekly_Sales"})

        ids = preds["unique_id"].str.split("_", n=1, expand=True)
        preds["Store"] = ids[0].astype(int)
        preds["Dept"] = ids[1].astype(int)
        preds["Id"] = (
            preds["Store"].astype(str) + "_" +
            preds["Dept"].astype(str) + "_" +
            preds["ds"].dt.strftime("%Y-%m-%d")
        )
        return preds[["Id", "Weekly_Sales"]]


##  Final fit on full history and log `TFT_Final`

Refits on all available train data with a fresh model instance, using
`BEST_HP_KWARGS` from the hyperparameter search and `h=HORIZON` (39, the
full test horizon - CV/HP folds only forecast 13 weeks). `start_padding_enabled=True`
means no series needs to be dropped for insufficient history, unlike the old
Darts notebook's `MIN_REQUIRED_LENGTH` filter. Wraps the fitted model, and
logs the wrapper (not the bare model) as the run artifact so
`model_inference.ipynb` can load it directly. A training-loss curve is
logged as an artifact on this run too.


In [ ]:
final_csv_logger = CSVLogger(save_dir="../submissions/_tft_logs", name="final")

final_model = TFT(
    h=HORIZON,
    input_size=INPUT_SIZE,
    stat_exog_list=STATIC_COLS,
    hist_exog_list=PAST_COVARIATE_COLS,
    futr_exog_list=FUTURE_COVARIATE_COLS,
    loss=MQLoss(quantiles=QUANTILES),
    max_steps=FINAL_MAX_STEPS,
    batch_size=BATCH_SIZE_FINAL,
    scaler_type="standard",
    random_seed=42,
    start_padding_enabled=True,
    **BEST_HP_KWARGS,
    **make_trainer_kwargs(logger=final_csv_logger),
)
final_nf = NeuralForecast(models=[final_model], freq=FREQ)

t0 = time.time()
final_nf.fit(
    df=panel_df[["unique_id", "ds", "y"] + PAST_COVARIATE_COLS + FUTURE_COVARIATE_COLS],
    static_df=static_df,
)
final_fit_seconds = time.time() - t0
print(f"Final fit took {final_fit_seconds:.1f}s over {panel_df[\'unique_id\'].nunique()} series")

pipeline = NeuralForecastPipeline(
    nf_model=final_nf,
    preprocessor=preprocessor,
    df_scaler=df_scaler,
    numeric_cols=numeric_cols,
    future_covariate_cols=FUTURE_COVARIATE_COLS,
    static_df=static_df,
    train_history_df=train_feat,
    freq=FREQ,
)

with mlflow.start_run(run_name="TFT_Final") as final_run:
    mlflow.log_params({
        "library": "neuralforecast",
        "max_steps": FINAL_MAX_STEPS,
        "batch_size": BATCH_SIZE_FINAL,
        "input_size": INPUT_SIZE,
        "horizon": HORIZON,
        "future_covariates": ",".join(FUTURE_COVARIATE_COLS) if FUTURE_COVARIATE_COLS else "none",
        "past_covariates": ",".join(PAST_COVARIATE_COLS),
        "static_covariates": ",".join(STATIC_COLS),
        "n_series_final": panel_df["unique_id"].nunique(),
        "cv_strategy": "expanding_window",
        "n_splits": N_SPLITS,
        "val_weeks": VAL_WEEKS,
        **BEST_HP_KWARGS,
    })
    mlflow.log_metric("cv_wmae_mean_at_selection", wmae_mean)
    mlflow.log_metric("cv_wmae_std_at_selection", wmae_std)
    mlflow.log_metric("cv_mae_mean_at_selection", mae_mean)
    mlflow.log_metric("final_fit_seconds", final_fit_seconds)

    try:
        final_loss_history = pd.read_csv(f"{final_csv_logger.log_dir}/metrics.csv")
        loss_col = "train_loss_epoch" if "train_loss_epoch" in final_loss_history else "train_loss"
        loss_curve = final_loss_history.dropna(subset=[loss_col]) if loss_col in final_loss_history else None
        plt.figure(figsize=(6, 4))
        if loss_curve is not None and not loss_curve.empty:
            plt.plot(loss_curve["step"], loss_curve[loss_col], marker="o", color="darkorange")
        plt.title("TFT Final - training loss")
        plt.xlabel("Step")
        plt.ylabel("Train loss")
        plt.tight_layout()
        plt.savefig("../submissions/_tft_final_train_loss.png", dpi=120)
        plt.close()
        mlflow.log_artifact("../submissions/_tft_final_train_loss.png")
    except Exception as e:
        print(f"Could not read/plot final loss history ({e})")

    with tempfile.TemporaryDirectory() as tmp:
        pkl_path = f"{tmp}/tft_pipeline.pkl"
        with open(pkl_path, "wb") as f:
            pickle.dump(pipeline, f)
        mlflow.log_artifact(pkl_path, artifact_path="pipeline")

    final_run_id = final_run.info.run_id

print("Logged TFT_Final wrapper artifact, run_id:", final_run_id)


##  Generate submission CSV

Reconciles against `sampleSubmission.csv` exactly - drops any predicted rows
beyond what's required, fills any missing ones (series `neuralforecast` had
no history for at all) with a store/dept mean fallback, so the row count and
ID set match precisely. Also pushes the submission CSV and the diagnostics
PNGs to MLflow as artifacts on `TFT_Final`.


In [ ]:
submission = pipeline.predict(test_raw)

sample = pd.read_csv("../data/raw/sampleSubmission.csv")
required_ids = set(sample["Id"])

extra_ids = set(submission["Id"]) - required_ids
print("Extra rows generated beyond what's needed:", len(extra_ids))

submission_filtered = submission[submission["Id"].isin(required_ids)].copy()
missing_ids = required_ids - set(submission_filtered["Id"])
print("Still missing after filtering:", len(missing_ids))

fallback_lookup = train_feat.groupby(["Store", "Dept"])["Weekly_Sales"].mean().to_dict()
global_fallback = train_feat["Weekly_Sales"].mean()

missing_df = pd.DataFrame({"Id": sorted(missing_ids)})
parts = missing_df["Id"].str.rsplit("_", n=1, expand=True)
store_dept = parts[0].str.split("_", n=1, expand=True)
missing_df["Store"] = store_dept[0].astype(int)
missing_df["Dept"] = store_dept[1].astype(int)
missing_df["Weekly_Sales"] = missing_df.apply(
    lambda r: fallback_lookup.get((r["Store"], r["Dept"]), global_fallback), axis=1
)

final_submission = pd.concat(
    [submission_filtered, missing_df[["Id", "Weekly_Sales"]]], ignore_index=True
).sort_values("Id").reset_index(drop=True)

print("Final rows:", len(final_submission), "| Expected:", len(sample))
assert len(final_submission) == len(sample)
assert set(final_submission["Id"]) == required_ids

final_submission.to_csv("../submissions/tft_submission.csv", index=False)
final_submission.head()


In [ ]:
with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact("../submissions/tft_submission.csv")
    mlflow.log_artifact("../submissions/_tft_diagnostics.png")
    mlflow.log_artifact("../submissions/_tft_wmae_by_type.png")
    mlflow.log_artifact("../submissions/_tft_representative_forecasts.png")

print("Submission + diagnostics logged as artifacts on TFT_Final (run_id:", final_run_id, ")")
